# Messages and Structured Output

In this exercise, you will learn two fundamental concepts in LangChain:

1. **Messages** - How we communicate with language models (LLMs)
2. **Structured Output** - How to get the model to return data in a specific, predictable format

By the end of this exercise, you will be able to:
- Understand the four message types: `HumanMessage`, `SystemMessage`, `AIMessage`, and `ToolMessage`
- Use **Pydantic** and **TypedDict** to define output schemas
- Extract structured data from unstructured text

## Setup

Make sure you have a `.env` file in your project directory with the following variables:

```
OPENAI_API_KEY=your-api-key-here
OPENAI_ENDPOINT=your-endpoint-here
```

Run the cell below to load them and verify everything is working.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Verify that the required environment variables are set
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY is not set! Check your .env file."
assert os.environ.get("OPENAI_ENDPOINT"), "OPENAI_ENDPOINT is not set! Check your .env file."

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(base_url=os.environ["OPENAI_ENDPOINT"], model="gpt-5.4-mini")

print("Environment variables loaded successfully!")

In [ ]:
# Quick test - make sure the model is working
response = llm.invoke("Say hello in one sentence!")
print(response.content)

---

## Part 1: Messages

When you use a chatbot, you type a message, the chatbot responds, and so on. Under the hood, this conversation is stored as a list of **messages**. Each message has:
- A **role** - who is "talking"
- **Content** - what they are saying

LangChain provides four main message types:

| Message Type | Who creates it | Purpose |
|---|---|---|
| `SystemMessage` | You (the developer) | Sets the model's behavior and personality |
| `HumanMessage` | You (the user) | Your question or input to the model |
| `AIMessage` | The model | The model's response |
| `ToolMessage` | A tool/function | The result of a tool the model called |

Let's look at each one.

### HumanMessage

A `HumanMessage` is what **you** send to the model. Think of it as typing a message in a chat window.

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# Create a simple agent using our model
agent = create_agent(model=llm, system_prompt="You are a friendly assistant. Keep answers short.")

# Create a HumanMessage
msg = HumanMessage("What is the capital of Denmark?")

# Send it to the agent
result = agent.invoke({"messages": [msg]})

# The model's response is the last message in the list
print(result["messages"][-1].content)

### SystemMessage

A `SystemMessage` gives the model instructions about **how it should behave**. Think of it like giving someone a job description before they start working.

When using `create_agent`, the system message is set with the `system_prompt` parameter.

Let's see how different system prompts change the model's behavior:

In [ ]:
# A cooking assistant
chef_agent = create_agent(
    model=llm,
    system_prompt="You are a professional chef. Give short, practical cooking advice.",
)

result = chef_agent.invoke({"messages": [HumanMessage("How do I make pasta?")]})
print("Chef says:")
print(result["messages"][-1].content)

In [ ]:
# A pirate!
pirate_agent = create_agent(
    model=llm,
    system_prompt="You are a pirate. Answer everything like a pirate would. Keep it to 2-3 sentences.",
)

result = pirate_agent.invoke({"messages": [HumanMessage("How do I make pasta?")]})
print("Pirate says:")
print(result["messages"][-1].content)

Same question, completely different answers! The `SystemMessage` controls the model's personality and behavior.

### AIMessage

An `AIMessage` is the model's response. You don't create these yourself - the model generates them.

Let's look at a conversation and see how `HumanMessage` and `AIMessage` appear together:

In [ ]:
result = agent.invoke({"messages": [HumanMessage("Tell me a fun fact about cats.")]})

# Loop through all messages and print their type and content
for msg in result["messages"]:
    print(f"Type: {msg.type}")
    print(f"Content: {msg.content}")
    print()

Notice how the result contains both your `human` message and the model's `ai` message. The full conversation history is stored in `result["messages"]`.

### Shortcut: Strings and Dictionaries

You don't always need to create `HumanMessage` objects explicitly. LangChain also accepts:
- **Strings** - automatically converted to a `HumanMessage`
- **Dictionaries** - with `role` and `content` keys

In [ ]:
# Using a plain string (becomes a HumanMessage automatically)
result = agent.invoke({"messages": "What color is the sky?"})
print(result["messages"][-1].content)

In [ ]:
# Using a dictionary
result = agent.invoke({"messages": {"role": "user", "content": "What color is grass?"}})
print(result["messages"][-1].content)

All three ways (`HumanMessage`, string, dictionary) do the same thing. Use whichever you find most readable!

### ToolMessage

When a model uses a **tool** (a Python function it can call), the tool's result comes back as a `ToolMessage`. This lets the model see the output and use it in its response.

Let's create a simple tool and watch how the messages flow:

In [ ]:
from langchain_core.tools import tool


@tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together and return the result."""
    return a + b


# Create an agent that can use our tool
math_agent = create_agent(
    model=llm,
    tools=[add_numbers],
    system_prompt="You are a math assistant. Use the add_numbers tool when asked to add.",
)

result = math_agent.invoke({"messages": "What is 15 + 27?"})

# pretty_print() shows messages in a nicely formatted way
for msg in result["messages"]:
    msg.pretty_print()

### Messages Summary

Notice the flow in the example above:
1. **HumanMessage** - You asked "What is 15 + 27?"
2. **AIMessage** - The model decided to call the `add_numbers` tool (with `a=15, b=27`)
3. **ToolMessage** - The tool returned the result (`42`)
4. **AIMessage** - The model used the tool result to give you the final answer

This is how all LangChain agents work: messages flow back and forth between you, the model, and any tools.

---

## Part 2: Structured Output

Sometimes we don't just want the model to write free text - we want it to return data in a **specific format** that our code can use reliably.

For example, imagine you have a paragraph about a movie and you want to extract the title, rating, and a summary. Without structured output, the model might format the answer differently each time. With structured output, you define a **schema** (a template) and the model **must** follow it.

LangChain supports two main ways to define schemas: **Pydantic** and **TypedDict**.

### What is Pydantic?

**Pydantic** is a popular Python library for defining data structures. Think of it like creating a **form** that data must fill out correctly.

You define a `class` that inherits from `BaseModel`, and list the fields with their types:

```python
from pydantic import BaseModel

class MovieReview(BaseModel):
    title: str        # text (a "string")
    rating: int       # a whole number (an "integer")
    summary: str      # text
```

This tells Python (and the LLM): *"A MovieReview must have a title (text), a rating (number), and a summary (text)."*

Key features:
- **Validates data** - if you try to put text where a number should be, it raises an error
- **Returns a Python object** - you access fields with a dot: `review.title`, `review.rating`
- **Most popular choice** for structured output in LangChain

In [ ]:
from pydantic import BaseModel


class MovieReview(BaseModel):
    title: str
    rating: int
    summary: str


# Create an agent with structured output
review_agent = create_agent(model=llm, response_format=MovieReview)

text = """I just watched Inception last night. What an incredible film!
The way Nolan plays with the concept of dreams within dreams is brilliant.
I'd give it a solid 9 out of 10. A must-watch for anyone who loves sci-fi thrillers."""

result = review_agent.invoke({"messages": [{"role": "user", "content": text}]})

# The structured response is a Pydantic object - use DOT notation
review = result["structured_response"]
print(f"Title: {review.title}")
print(f"Rating: {review.rating}")
print(f"Summary: {review.summary}")
print()
print(f"Type: {type(review)}")

The model extracted exactly the fields we asked for, in the types we specified. No matter how many times you run this, the output will always have the same structure.

### What is TypedDict?

**TypedDict** is built into Python (no extra library needed). It defines the expected shape of a dictionary.

```python
from typing import TypedDict

class MovieReview(TypedDict):
    title: str
    rating: int
    summary: str
```

This says: *"A MovieReview is a dictionary with keys `title` (text), `rating` (number), and `summary` (text)."*

Key differences from Pydantic:
- **Returns a plain dictionary** - you access fields with brackets: `review["title"]`
- **No validation** - it won't raise an error if types don't match
- **Simpler** - no extra library needed, just Python's built-in `typing` module

In [ ]:
from typing import TypedDict


class MovieReviewDict(TypedDict):
    title: str
    rating: int
    summary: str


# Create an agent with TypedDict structured output
review_agent_dict = create_agent(model=llm, response_format=MovieReviewDict)

text = """Saw The Matrix again yesterday. Still holds up after all these years.
The action scenes are legendary and the philosophy is thought-provoking.
I'd rate it 8 out of 10."""

result = review_agent_dict.invoke({"messages": [{"role": "user", "content": text}]})

# The structured response is a dictionary - use BRACKET notation
review = result["structured_response"]
print(f"Title: {review['title']}")
print(f"Rating: {review['rating']}")
print(f"Summary: {review['summary']}")
print()
print(f"Type: {type(review)}")

### Pydantic vs TypedDict

| Feature | Pydantic (`BaseModel`) | `TypedDict` |
|---|---|---|
| Import | `from pydantic import BaseModel` | `from typing import TypedDict` |
| Returns | Python object (dot: `obj.field`) | Dictionary (brackets: `obj["field"]`) |
| Validation | Yes - checks types automatically | No |
| Extra library needed | Yes (`pydantic`) | No (built into Python) |
| Best for | Production apps, complex schemas | Quick prototypes, simple schemas |

Both work with `create_agent(response_format=...)` in exactly the same way.

### The model always follows the schema

An important property of structured output: the model **must** return data matching your schema, even if the input has nothing to do with the schema. Watch what happens:

In [ ]:
# Asking a completely unrelated question with the MovieReview schema
result = review_agent.invoke({"messages": "How much is 2 + 2?"})
print(result["structured_response"])

Even though the question had nothing to do with movies, the model still returned a `MovieReview` object. The schema is enforced at the API level - it's not a suggestion, it's a constraint.

---

## Part 3: Exercises

Now it's your turn! Complete the exercises below by replacing the `TODO` comments with your code.

### Exercise 1: Working with Messages

Create an agent with a fun system prompt of your choice. Send it a `HumanMessage` and print the response. Then, loop through all the messages and print each message's type and content.

In [ ]:
# TODO: Create an agent with a creative system prompt
# agent = create_agent(model=llm, system_prompt="Your system prompt here")

# TODO: Send a HumanMessage to the agent
# result = agent.invoke({"messages": [HumanMessage("Your question here")]})

# TODO: Print the model's response (the last message)
# print(result["messages"][-1].content)

# TODO: Loop through ALL messages and print their type and content
# for msg in result["messages"]:
#     print(f"Type: {msg.type}")
#     print(f"Content: {msg.content}")
#     print()

### Exercise 2: System Messages Change Behavior

Create **two different agents** with different system prompts. Send the **same question** to both and compare how their responses differ.

For example, ask both *"What should I eat for dinner?"* but make one a health coach and the other a pizza enthusiast.

In [ ]:
# TODO: Create the first agent with a specific personality
# agent1 = create_agent(model=llm, system_prompt="...")

# TODO: Create the second agent with a different personality
# agent2 = create_agent(model=llm, system_prompt="...")

# TODO: Ask both agents the same question
# question = "Your question here"
# result1 = agent1.invoke({"messages": question})
# result2 = agent2.invoke({"messages": question})

# TODO: Print both responses and compare
# print("Agent 1 says:")
# print(result1["messages"][-1].content)
# print("\nAgent 2 says:")
# print(result2["messages"][-1].content)

### Exercise 3: Structured Output with Pydantic

Define a Pydantic `BaseModel` for a **recipe** with these fields:
- `name` (str) - the name of the recipe
- `servings` (int) - how many people it serves
- `ingredients` (list[str]) - a list of ingredients

Then create an agent that extracts this information from a text description. Remember: Pydantic returns an object, so use **dot notation** (e.g., `recipe.name`).

In [ ]:
# TODO: Define your Pydantic schema
# class Recipe(BaseModel):
#     name: str
#     servings: int
#     ingredients: list[str]

# TODO: Create an agent with response_format=Recipe
# agent = create_agent(model=llm, response_format=Recipe)

# TODO: Write a text description of a recipe
# text = "Here is my grandma's pancake recipe for 4 people. You need flour, eggs, milk, sugar, and butter..."

# TODO: Invoke the agent and print the structured response
# result = agent.invoke({"messages": [{"role": "user", "content": text}]})
# recipe = result["structured_response"]
# print(f"Recipe: {recipe.name}")
# print(f"Servings: {recipe.servings}")
# print(f"Ingredients: {recipe.ingredients}")

### Exercise 4: Structured Output with TypedDict

Define a `TypedDict` for a **weather report** with these fields:
- `city` (str)
- `temperature` (int)
- `condition` (str) - e.g., "sunny", "rainy", "cloudy"

Create an agent and extract weather information from a text description. Remember: TypedDict returns a **dictionary**, so use **bracket notation** (e.g., `report["city"]`).

In [ ]:
# TODO: Define your TypedDict schema
# class WeatherReport(TypedDict):
#     city: str
#     temperature: int
#     condition: str

# TODO: Create an agent with response_format=WeatherReport
# agent = create_agent(model=llm, response_format=WeatherReport)

# TODO: Write a text about the weather
# text = "It's a beautiful day in Copenhagen! The sun is shining and it's about 22 degrees outside."

# TODO: Invoke the agent and print the structured response
# result = agent.invoke({"messages": [{"role": "user", "content": text}]})
# report = result["structured_response"]
# print(f"City: {report['city']}")
# print(f"Temperature: {report['temperature']}")
# print(f"Condition: {report['condition']}")

### Bonus: Combine Messages and Structured Output

Create an agent that has both a **system prompt** and a **structured output format**. For example, create a "book reviewer" agent that always returns reviews in a specific schema.

Hint: `create_agent` accepts both `system_prompt` and `response_format` at the same time!

In [ ]:
# TODO: Define a schema and create an agent with both system_prompt and response_format
